# Netflix Titles: Dataset Overview
First look at the **raw** dataset (`netflix_titles.csv`), followed by the genre, release-year and country views the dashboard will be built on. Run all cells before saving so the outputs are kept.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display plots inside the notebook
%matplotlib inline
sns.set_theme(style="whitegrid")

## 1. Making a dataframe from the dataset

In [ ]:
df = pd.read_csv("../data/netflix_titles.csv")

## 2. Glance at the first few entries

In [ ]:
df.head()

## 3. Shape and column names

In [ ]:
print("Shape:", df.shape)
print("Columns:", list(df.columns))

## 4. Summary of the dataset

In [ ]:
df.info()

## 5. Statistical summary for the entire dataframe

In [ ]:
df.describe(include="all")

## 6. Checking the number of null values

In [ ]:
nulls = pd.DataFrame({
    "Missing": df.isnull().sum(),
    "Missing %": (df.isnull().mean() * 100).round(2),
}).sort_values("Missing", ascending=False)
nulls

## 7. Checking for duplicates
`show_id` is unique, so also check `title + type + release_year`.

In [ ]:
print("Full-row duplicates:        ", df.duplicated().sum())
print("title/type/year duplicates: ", df.duplicated(subset=["title", "type", "release_year"]).sum())

## 8. Content type split

In [ ]:
print(df["type"].unique())
df["type"].value_counts()

## 9. Rating counts
Some rows contain a duration such as `74 min` in `rating`. This is a data-entry error in the source file. The cleaning notebook moves these values back into `duration`.

In [ ]:
df["rating"].value_counts()

In [ ]:
df[df["rating"].str.contains("min", na=False)][["show_id", "title", "rating", "duration"]]

## 10. Data types, missing values and unique values per column

In [ ]:
summary = pd.DataFrame({
    "Data Type": df.dtypes,
    "Missing Values": df.isnull().sum(),
    "Unique Values": df.nunique()
})
summary

## 11. Multi-value columns
`country`, `listed_in`, `cast` and `director` hold comma-separated lists, so a plain `value_counts()` treats `"United States, India"` as its own category. The counts below show the difference.

In [ ]:
print("Raw top 5 countries (misleading):")
print(df["country"].value_counts().head(5))

In [ ]:
def explode_column(frame, source_col, new_col):
    out = frame[["show_id", "type", "release_year", source_col]].dropna(subset=[source_col]).copy()
    out[new_col] = out[source_col].str.split(",")
    out = out.explode(new_col)
    out[new_col] = out[new_col].str.strip()
    return out[out[new_col] != ""].drop(columns=source_col)

genres = explode_column(df, "listed_in", "genre")
countries = explode_column(df, "country", "country_single")

print("Distinct genres:   ", genres["genre"].nunique())
print("Distinct countries:", countries["country_single"].nunique())

## 12. Top 10 countries (after splitting)

In [ ]:
top_countries = countries["country_single"].value_counts().head(10)
print(top_countries)

plt.figure(figsize=(9, 5))
sns.barplot(x=top_countries.values, y=top_countries.index, color="#E50914")
plt.title("Top 10 Content-Producing Countries")
plt.xlabel("Number of titles")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 13. Top 10 genres (after splitting)

In [ ]:
top_genres = genres["genre"].value_counts().head(10)
print(top_genres)

plt.figure(figsize=(9, 5))
sns.barplot(x=top_genres.values, y=top_genres.index, color="#E50914")
plt.title("Top 10 Genres")
plt.xlabel("Number of titles")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 14. Release-year distribution

In [ ]:
plt.figure(figsize=(11, 4.5))
sns.histplot(data=df, x="release_year", hue="type", multiple="stack", binwidth=2,
             palette={"Movie": "#E50914", "TV Show": "#221F1F"})
plt.title("Titles by Release Year")
plt.xlabel("Release year")
plt.ylabel("Number of titles")
plt.tight_layout()
plt.show()

print("Release year range:", df["release_year"].min(), "to", df["release_year"].max())
print("Share of titles released 2015 or later:", round((df["release_year"] >= 2015).mean() * 100, 1), "%")

## 15. Movies vs TV shows by release year (recent years)

In [ ]:
recent = df[df["release_year"] >= 2000].groupby(["release_year", "type"]).size().unstack(fill_value=0)

recent.plot(figsize=(11, 4.5), marker="o", color={"Movie": "#E50914", "TV Show": "#221F1F"})
plt.title("Movies vs TV Shows by Release Year (2000 onwards)")
plt.xlabel("Release year")
plt.ylabel("Number of titles")
plt.tight_layout()
plt.show()

## 16. Findings to carry into the dashboard
Fill these in after running the notebook:
- Content split between movies and TV shows:
- Top genres and countries:
- Years with the most releases:
- Data issues the cleaning notebook must handle: missing `director`, `country` and `cast`; duration stored in `rating`; multi-value columns